# Seal OCR Recognition v3 — Kaggle
Attach the private `seal-recognition-v3-20k` dataset, enable Internet and one GPU. Run `control` and `primary` in separate sessions; only run tuning experiments when both miss the acceptance gate.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, sys, urllib.request
EXPERIMENT = 'primary'  # control | primary | spacing-050 | spacing-025
SMOKE_EPOCHS = 0       # set to 1 for the mandatory smoke run
PROBABILITIES = {'control': 0.0, 'primary': 0.35, 'spacing-050': 0.50, 'spacing-025': 0.25}
if EXPERIMENT not in PROBABILITIES: raise ValueError(EXPERIMENT)
SPACING_PROB = PROBABILITIES[EXPERIMENT]
SEED = 20260919
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
WORK = Path('/kaggle/working/seal-ocr-v3')
RUN_DIR = Path('/kaggle/working/runs') / f'seal-ocr-rec-v3-{EXPERIMENT}'
PADDLE = Path('/kaggle/working/PaddleOCR')
for folder in (WORK, RUN_DIR):
    if folder.exists(): shutil.rmtree(folder)
    folder.mkdir(parents=True)
print({'experiment': EXPERIMENT, 'spacingProbability': SPACING_PROB, 'smokeEpochs': SMOKE_EPOCHS})

In [ ]:
# Discover and verify the immutable private dataset.
candidates = []
for metadata_path in Path('/kaggle/input').rglob('metadata.json'):
    try:
        metadata = json.loads(metadata_path.read_text())
        if metadata.get('datasetVersion') == 'seal-recognition-v3': candidates.append((metadata_path.parent, metadata))
    except Exception: pass
if len(candidates) != 1: raise RuntimeError(f'Expected one seal-recognition-v3 input, found {len(candidates)}')
DATASET, metadata = candidates[0]
if (metadata.get('totalVerified'), metadata.get('trainSamples'), metadata.get('valSamples')) != (14500, 13000, 1500):
    raise RuntimeError(f'Unexpected dataset counts: {metadata}')
rows = list(csv.DictReader((DATASET / 'manifest.csv').open(encoding='utf-8-sig')))
if len(rows) != 14500: raise RuntimeError(f'Manifest count: {len(rows)}')
fingerprint_lines = []
for row in rows:
    image = DATASET / row['image']
    digest = hashlib.sha256(image.read_bytes()).hexdigest()
    if digest != row['sha256']: raise RuntimeError(f'Hash mismatch: {row["image"]}')
    fingerprint_lines.append(f"{row['image']}\t{row['ground_truth']}\t{digest}")
fingerprint = hashlib.sha256('\n'.join(sorted(fingerprint_lines)).encode()).hexdigest()
if fingerprint != metadata['datasetSha256']: raise RuntimeError('Dataset fingerprint mismatch')
print({'dataset': str(DATASET), 'sha256': fingerprint, 'rows': len(rows)})

In [ ]:
# Pin the training stack.
if not PADDLE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'v3.7.0', 'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLE)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'paddlepaddle-gpu==3.2.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PADDLE / 'requirements.txt'), 'pyyaml'], check=True)
import paddle
assert paddle.device.is_compiled_with_cuda()
paddle.utils.run_check()
print({'paddle': paddle.__version__, 'device': paddle.device.get_device()})

In [ ]:
# Install the tested train-only transform into the pinned PaddleOCR checkout.
transform_source = r'''import random
import cv2
from .text_image_aug import tia_stretch

class RandomCharacterSpacing:
    def __init__(self, prob=0.35, min_segments=3, max_segments=6, **kwargs):
        self.prob=float(prob); self.min_segments=int(min_segments); self.max_segments=int(max_segments)
    def __call__(self, data):
        image=data["image"]
        if random.random() > self.prob or image.shape[0] < 20 or image.shape[1] < 20: return data
        height=image.shape[0]
        stretched=tia_stretch(image, random.randint(self.min_segments, self.max_segments))
        if stretched.shape[0] != height:
            stretched=cv2.resize(stretched, (stretched.shape[1], height), interpolation=cv2.INTER_CUBIC)
        data["image"]=stretched
        return data
'''
target = PADDLE / 'ppocr/data/imaug/random_character_spacing.py'
target.write_text(transform_source)
init_path = PADDLE / 'ppocr/data/imaug/__init__.py'
import_line = 'from .random_character_spacing import RandomCharacterSpacing\n'
if import_line not in init_path.read_text(): init_path.write_text(init_path.read_text() + '\n' + import_line)
print({'transformSha256': hashlib.sha256(target.read_bytes()).hexdigest()})

In [ ]:
# Build the v3 recipe from the official pinned config.
import yaml
source_config = WORK / 'official.yml'
urllib.request.urlretrieve('https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/v3.7.0/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml', source_config)
config = yaml.safe_load(source_config.read_text())
config['Global'].update({'epoch_num': SMOKE_EPOCHS or 60, 'seed': SEED, 'save_model_dir': str(RUN_DIR / 'checkpoints'), 'save_epoch_step': 10, 'eval_batch_step': [0, 100], 'distributed': False, 'use_visualdl': False, 'max_text_length': 25})
config['Optimizer']['lr'].update({'learning_rate': 0.00005, 'warmup_epoch': 2})
transforms = [
 {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
 {'RecAug': {'tia_prob': 0.10, 'crop_prob': 0.0, 'reverse_prob': 0.0, 'noise_prob': 0.15, 'jitter_prob': 0.05, 'blur_prob': 0.10, 'hsv_aug_prob': 0.15}},
]
if SPACING_PROB: transforms.append({'RandomCharacterSpacing': {'prob': SPACING_PROB, 'min_segments': 3, 'max_segments': 6}})
transforms += [{'MultiLabelEncode': {'gtc_encode': 'NRTRLabelEncode'}}, {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}}]
config['Train']['dataset'].update({'data_dir': str(DATASET), 'label_file_list': [str(DATASET / 'train.txt')], 'transforms': transforms, 'ratio_list': [1.0]})
config['Train']['loader'].update({'batch_size_per_card': 32, 'drop_last': False, 'num_workers': 2})
config['Eval']['dataset'].update({'data_dir': str(DATASET), 'label_file_list': [str(DATASET / 'val.txt')]})
config['Eval']['loader'].update({'batch_size_per_card': 64, 'num_workers': 2})
CONFIG = RUN_DIR / 'config.yml'
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
PRETRAIN = RUN_DIR / 'PP-OCRv6_medium_rec_pretrained.pdparams'
urllib.request.urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams', PRETRAIN)
print(CONFIG)

In [ ]:
# Train and export the best checkpoint.
subprocess.run([sys.executable, 'tools/train.py', '-c', str(CONFIG), '-o', f'Global.pretrained_model={PRETRAIN}', f'Global.save_model_dir={RUN_DIR / "checkpoints"}'], cwd=PADDLE, check=True)
BEST = RUN_DIR / 'checkpoints/best_accuracy'
INFERENCE = RUN_DIR / 'inference'
subprocess.run([sys.executable, 'tools/export_model.py', '-c', str(CONFIG), '-o', f'Global.pretrained_model={BEST}', f'Global.save_inference_dir={INFERENCE}'], cwd=PADDLE, check=True)
run_metadata = {'modelVersion': 'seal-ocr-rec-v3', 'experiment': EXPERIMENT, 'spacingProbability': SPACING_PROB, 'datasetSha256': metadata['datasetSha256'], 'seed': SEED, 'epochs': SMOKE_EPOCHS or 60}
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2))
archive = shutil.make_archive(f'/kaggle/working/seal-ocr-rec-v3-{EXPERIMENT}', 'zip', RUN_DIR)
print({'archive': archive, 'sha256': hashlib.sha256(Path(archive).read_bytes()).hexdigest()})